In [15]:
!pip install pyspark

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import collect_set as F_collect_set
import pyspark.sql.functions as F
from pyspark.ml.fpm import FPGrowth
from itertools import combinations
import pandas as pd

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
sp = SparkSession.builder.appName('baskets').getOrCreate()
data = sp.read.csv('/content/drive/My Drive/BIG DATA/baskets.csv', header = True)

In [19]:
data.head(5)

[Row(Member_number='1249', Date='01/01/2014', itemDescription='citrus fruit', year='2014', month='1', day='1', day_of_week='2'),
 Row(Member_number='1249', Date='01/01/2014', itemDescription='coffee', year='2014', month='1', day='1', day_of_week='2'),
 Row(Member_number='1381', Date='01/01/2014', itemDescription='curd', year='2014', month='1', day='1', day_of_week='2'),
 Row(Member_number='1381', Date='01/01/2014', itemDescription='soda', year='2014', month='1', day='1', day_of_week='2'),
 Row(Member_number='1440', Date='01/01/2014', itemDescription='other vegetables', year='2014', month='1', day='1', day_of_week='2')]

In [20]:
data.describe()

DataFrame[summary: string, Member_number: string, Date: string, itemDescription: string, year: string, month: string, day: string, day_of_week: string]

In [21]:
data.printSchema()

root
 |-- Member_number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- itemDescription: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day: string (nullable = true)
 |-- day_of_week: string (nullable = true)



In [22]:
data.show(5)

+-------------+----------+----------------+----+-----+---+-----------+
|Member_number|      Date| itemDescription|year|month|day|day_of_week|
+-------------+----------+----------------+----+-----+---+-----------+
|         1249|01/01/2014|    citrus fruit|2014|    1|  1|          2|
|         1249|01/01/2014|          coffee|2014|    1|  1|          2|
|         1381|01/01/2014|            curd|2014|    1|  1|          2|
|         1381|01/01/2014|            soda|2014|    1|  1|          2|
|         1440|01/01/2014|other vegetables|2014|    1|  1|          2|
+-------------+----------+----------------+----+-----+---+-----------+
only showing top 5 rows



In [23]:
print(data.count())
data.distinct().count()


38765


38006

In [24]:
data.select('itemDescription').distinct().count()

167

In [25]:
items = data.select('itemDescription').distinct()
items.show(5)

+------------------+
|   itemDescription|
+------------------+
|pickled vegetables|
|         beverages|
|    snack products|
|           vinegar|
|            dishes|
+------------------+
only showing top 5 rows



In [30]:
def main():

    spark = SparkSession.builder.appName("Task3").getOrCreate()


    apriori_df = pd.read_csv(
        '/content/drive/My Drive/BIG DATA/pass2_output.csv',
        header=None,
        names=['item1', 'item2', 'support']
    )

    frequent_itemsets_apriori = set()
    for _, row in apriori_df.iterrows():
        itemset = frozenset([str(row['item1']).strip(), str(row['item2']).strip()])
        frequent_itemsets_apriori.add(itemset)
    num_frequent_itemsets_apriori = len(frequent_itemsets_apriori)

    num_rules_apriori = len(apriori_df)

    print(f"Số lượng tập phổ biến (Apriori): {num_frequent_itemsets_apriori}")
    print(f"Số lượng luật kết hợp (Apriori): {num_rules_apriori}")

    pcy_df = pd.read_csv(
        '/content/drive/My Drive/BIG DATA/association_rules.csv',
        header=None,
        names=['raw_rule']
    )

    pcy_df[['rule', 'confidence']] = pcy_df['raw_rule'].str.extract(r'(.+?)\s*,\s*([\d.]+)')

    frequent_itemsets_pcy = set()
    for rule in pcy_df['rule']:
        if '=>' not in rule:
            continue
        antecedent, consequent = rule.split(' => ')
        antecedent_items = set(antecedent.strip().split(','))
        consequent_items = set(consequent.strip().split(','))
        itemset = frozenset(antecedent_items | consequent_items)
        frequent_itemsets_pcy.add(itemset)

    num_frequent_itemsets_pcy = len(frequent_itemsets_pcy)
    num_rules_pcy = len(pcy_df)

    print(f"Số lượng luật kết hợp (PCY): {num_rules_pcy}")
    print(f"Số lượng tập phổ biến (PCY): {num_frequent_itemsets_pcy}")

    data = spark.read.csv(
        '/content/drive/My Drive/BIG DATA/baskets.csv',
        header=True,
        inferSchema=True
    )

    tx = (
        data.groupBy("Member_number", "Date")
        .agg(F.collect_set("itemDescription").alias("items"))
        .filter(F.size("items") > 1)
    )

    print("Số lượng giao dịch:", tx.count())
    tx.show(5, truncate=False)

    fpGrowth = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.1)
    fp_model = fpGrowth.fit(tx)

    fp_freq = fp_model.freqItemsets.select("items").distinct()
    fp_rules = fp_model.associationRules.select(
        F.concat_ws(' => ',
                    F.array_join("antecedent", ","),
                    F.array_join("consequent", ",")
                   ).alias("rule")
    ).distinct()

    count_fp_freq = fp_freq.count()
    count_fp_rules = fp_rules.count()
    print(f"Số lượng tập phổ biến (FP-Growth): {count_fp_freq}")
    print(f"Số lượng luật kết hợp (FP-Growth): {count_fp_rules}")

    apriori_freq_spark = spark.createDataFrame(
        [(list(s),) for s in frequent_itemsets_apriori],
        ["items"]
    ).distinct()

    pcy_freq_spark = spark.createDataFrame(
        [(list(s),) for s in frequent_itemsets_pcy],
        ["items"]
    ).distinct()

    overlap_apriori_fp = apriori_freq_spark.intersect(fp_freq).count()
    overlap_pcy_fp = pcy_freq_spark.intersect(fp_freq).count()

    print(f"Tập phổ biến giao nhau với FP-Growth: Apriori={overlap_apriori_fp}, PCY={overlap_pcy_fp}")

    def percent_difference(current, standard):
        if standard == 0:
            return 0
        return ((current - standard) / standard) * 100

    count_fp = count_fp_freq

    freq_percent_apriori = percent_difference(num_frequent_itemsets_apriori, count_fp)
    freq_percent_pcy = percent_difference(num_frequent_itemsets_pcy, count_fp)
    print(f"\nFrequent itemsets count (% difference from FP-Growth): Apriori={freq_percent_apriori:.2f}%, PCY={freq_percent_pcy:.2f}%")

    rules_percent_apriori = percent_difference(num_rules_apriori, count_fp_rules)
    rules_percent_pcy = percent_difference(num_rules_pcy, count_fp_rules)
    print(f"Association rules count (% difference from FP-Growth): Apriori={rules_percent_apriori:.2f}%, PCY={rules_percent_pcy:.2f}%")

    overlap_apriori_percent = (overlap_apriori_fp / count_fp_freq) * 100 if count_fp_freq else 0
    overlap_pcy_percent = (overlap_pcy_fp / count_fp_freq) * 100 if count_fp_freq else 0

    print(f"Tỉ lệ giao nhau với FP-Growth (%): Apriori={overlap_apriori_percent:.2f}%, PCY={overlap_pcy_percent:.2f}%")

    print("\n=== Tổng kết ===")
    print(f"- Apriori: {num_frequent_itemsets_apriori} tập phổ biến, {num_rules_apriori} luật")
    print(f"- PCY: {num_frequent_itemsets_pcy} tập phổ biến, {num_rules_pcy} luật")
    print(f"- FP-Growth: {count_fp_freq} tập phổ biến, {count_fp_rules} luật")

    print("\n===  KẾT LUẬN TỔNG HỢP ===")

    # So sánh tập phổ biến
    if count_fp > num_frequent_itemsets_apriori and count_fp > num_frequent_itemsets_pcy:
        print(" FP-Growth tìm được nhiều tập phổ biến hơn cả Apriori và PCY.")
    elif count_fp < num_frequent_itemsets_apriori or count_fp < num_frequent_itemsets_pcy:
        print(" Apriori hoặc PCY tìm được nhiều tập phổ biến hơn FP-Growth (có thể do tham số hoặc cách đếm khác).")
    else:
        print(" Số lượng tập phổ biến của FP-Growth tương đương với Apriori và PCY.")

    # So sánh luật kết hợp
    if count_fp_rules > num_rules_apriori and count_fp_rules > num_rules_pcy:
        print(" FP-Growth sinh ra nhiều luật kết hợp hơn — chứng tỏ độ bao phủ và hiệu quả khai phá cao hơn.")
    elif count_fp_rules < num_rules_apriori or count_fp_rules < num_rules_pcy:
        print(" Apriori hoặc PCY sinh ra nhiều luật hơn — tuy nhiên có thể chứa các luật dư thừa hoặc trùng lặp.")
    else:
        print(" Số lượng luật kết hợp giữa các thuật toán gần tương đương nhau.")
 # So sánh độ trùng khớp
    if overlap_apriori_percent > overlap_pcy_percent:
        print(f" Apriori có {overlap_apriori_percent:.2f}% tập phổ biến trùng FP-Growth — cao hơn PCY ({overlap_pcy_percent:.2f}%).")
    elif overlap_apriori_percent < overlap_pcy_percent:
        print(f" PCY có {overlap_pcy_percent:.2f}% tập phổ biến trùng FP-Growth — cao hơn Apriori ({overlap_apriori_percent:.2f}%).")
    else:
        print(" Cả hai thuật toán Apriori và PCY có mức trùng khớp tương đương với FP-Growth.")

    # Tổng quan hiệu quả
    print("\n Đánh giá tổng quan:")
    print("- FP-Growth hoạt động nhanh và hiệu quả hơn trong xử lý dữ liệu lớn vì không cần sinh nhiều ứng viên.")
    print("- Apriori dễ hiểu nhưng chậm hơn vì phải duyệt nhiều vòng lặp.")
    print("- PCY cải tiến hơn Apriori nhờ dùng hashing giảm bộ nhớ, nhưng vẫn kém FP-Growth trong dữ liệu lớn.")
    print("\n Kết luận: FP-Growth là thuật toán hiệu quả nhất trong ba thuật toán khi xét về tốc độ và khả năng khai phá tập phổ biến lớn.")

if __name__ == "__main__":
    main()


Số lượng tập phổ biến (Apriori): 6260
Số lượng luật kết hợp (Apriori): 37686
Số lượng luật kết hợp (PCY): 131
Số lượng tập phổ biến (PCY): 131
Số lượng giao dịch: 14758
+-------------+----------+--------------------------------------------------+
|Member_number|Date      |items                                             |
+-------------+----------+--------------------------------------------------+
|1000         |15/03/2015|[whole milk, sausage, yogurt, semi-finished bread]|
|1000         |24/06/2014|[pastry, whole milk, salty snack]                 |
|1000         |24/07/2015|[misc. beverages, canned beer]                    |
|1000         |25/11/2015|[sausage, hygiene articles]                       |
|1000         |27/05/2015|[pickled vegetables, soda]                        |
+-------------+----------+--------------------------------------------------+
only showing top 5 rows

Số lượng tập phổ biến (FP-Growth): 750
Số lượng luật kết hợp (FP-Growth): 134
Tập phổ biến giao nhau với